# 09 · Fix the corrupted boolean columns in `docker_tickets_v5`

A one-shot patch, not a re-run of the pipeline. `is_synthetic`, `overlaps_kb`, and `has_code` read `FALSE` for
almost every row in your real file — a notebook 07 bug (fixed there for next time, but this file was already
written by the broken version). None of the *content* is affected — problem/resolution text, topics, trust
scores, sources are all correct — only these 3 columns need recomputing, so this doesn't repeat the forum scan
or anything slow. Takes seconds.

- `is_synthetic` → `source_type == "hf_synthetic_qa"` (that's the only source that's ever synthetic)
- `overlaps_kb` → `source_type == "official_docs_entry"` (the only source that duplicates the curated KB)
- `has_code` → whether `problem` or `resolution` actually contains a ` ``` ` fence

Upload `docker_tickets_v5.csv` when prompted. Writes `docker_tickets_v5_fixed.csv` / `.jsonl`.

In [ ]:
import csv
import json
from pathlib import Path

csv.field_size_limit(10_000_000)

try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass

try:
    from google.colab import files
    uploaded = files.upload()
    input_path = Path(next(iter(uploaded)))
except ImportError:
    input_path = Path("docker_tickets_v5.csv")  # local fallback

OUT_DIR = input_path.parent
print("Reading:", input_path.resolve())

In [ ]:
with open(input_path, encoding="utf-8-sig", newline="") as f:
    reader = csv.DictReader(f)
    fieldnames = reader.fieldnames
    rows = list(reader)

print("rows loaded:", len(rows))

before = {
    "is_synthetic_TRUE": sum(1 for r in rows if r["is_synthetic"] == "TRUE"),
    "overlaps_kb_TRUE": sum(1 for r in rows if r["overlaps_kb"] == "TRUE"),
    "has_code_TRUE": sum(1 for r in rows if r["has_code"] == "TRUE"),
}

for row in rows:
    row["is_synthetic"] = "TRUE" if row["source_type"] == "hf_synthetic_qa" else "FALSE"
    row["overlaps_kb"] = "TRUE" if row["source_type"] == "official_docs_entry" else "FALSE"
    row["has_code"] = "TRUE" if ("```" in row["problem"] or "```" in row["resolution"]) else "FALSE"

after = {
    "is_synthetic_TRUE": sum(1 for r in rows if r["is_synthetic"] == "TRUE"),
    "overlaps_kb_TRUE": sum(1 for r in rows if r["overlaps_kb"] == "TRUE"),
    "has_code_TRUE": sum(1 for r in rows if r["has_code"] == "TRUE"),
}

print("before:", before)
print("after: ", after)

## Save

In [ ]:
LIST_COLUMNS = ["trust_reasons", "topics", "tags", "docker_versions", "error_strings", "exit_codes"]

out_csv = OUT_DIR / "docker_tickets_v5_fixed.csv"
out_jsonl = OUT_DIR / "docker_tickets_v5_fixed.jsonl"

with out_csv.open("w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames, quoting=csv.QUOTE_ALL)
    writer.writeheader()
    writer.writerows(rows)

with out_jsonl.open("w", encoding="utf-8") as f:
    for row in rows:
        jsonl_row = dict(row)
        for col in LIST_COLUMNS:
            try:
                jsonl_row[col] = json.loads(jsonl_row[col])
            except json.JSONDecodeError:
                jsonl_row[col] = []
        for col in ("is_synthetic", "overlaps_kb", "has_code"):
            jsonl_row[col] = jsonl_row[col] == "TRUE"
        f.write(json.dumps(jsonl_row, ensure_ascii=False) + "\n")

print(f"Saved {len(rows)} rows to {out_csv.name} and {out_jsonl.name}")

try:
    from google.colab import files
    files.download(str(out_csv))
except ImportError:
    pass